# 02 — Characteristics: Grower vs Non-grower

In [1]:
from pathlib import Path
import sys

# Find the project root (parent of notebooks/) so absolute paths work
# from anywhere the notebook is launched.
PROJECT_ROOT = Path('/scratch/ctaylor/core_models_analysis')
REPORTS = PROJECT_ROOT / 'reports'
RESULTS = PROJECT_ROOT / 'results'
LOGS    = PROJECT_ROOT / 'logs'
DATA    = PROJECT_ROOT / 'data'
SCRIPTS = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS))

# KBUtils NotebookSession: opens .kbcache/ alongside this notebook so
# heavy outputs can be saved/loaded with provenance.
from kbutillib.notebook import NotebookSession
session = NotebookSession.for_notebook(notebook_file=__file__ if '__file__' in dir() else None,
                                       project_name='core_models_analysis')
print('Cache directory:', session.kbcache_dir)
print('Notebook name :', session.notebook_name)

Cache directory: /scratch/ctaylor/core_models_analysis/notebooks/.kbcache
Notebook name : None


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


## What this notebook does

Compares grower vs non-grower distributions across the size metrics
(metabolites, reactions, genes, exchanges) and the biomass-flux
bucket histogram, mirroring `scripts/deeper_analysis.py` part 1.

In [2]:
import pandas as pd
from collections import Counter
df = pd.read_csv(RESULTS / 'results.csv')
df['grows'] = df['grows'].astype(str).str.lower() == 'true'
keys = ['n_metabolites', 'n_reactions', 'n_genes',
        'n_exchanges_total', 'n_exchanges_open']
tbl = df.groupby('grows')[keys].agg(['median', 'mean']).T
tbl.columns = ['non-grower', 'grower']
tbl

non-grower      grower
n_metabolites     median  125.000000  158.000000
                  mean    124.294329  156.608495
n_reactions       median  118.000000  175.000000
                  mean    119.842034  171.718867
n_genes           median  101.000000  189.000000
                  mean    106.350135  193.655591
n_exchanges_total median   23.000000   27.000000
                  mean     23.309631   26.681017
n_exchanges_open  median   16.000000   19.000000
                  mean     16.122412   19.144178

In [3]:
# Cache the characteristics table
session.cache.save('characteristics_table', tbl.reset_index(),
                   type_hint='dataframe',
                   metadata={'description': 'grower vs non-grower size stats'})
print('cached as characteristics_table')

cached as characteristics_table


In [4]:
# Flux-bucket histogram
bins = [0, 1, 5, 10, 25, 50, 75, 100]
growers = df[df['grows']]
hist = []
prev = 0
for hi in bins[1:]:
    n = int(((growers['growth_flux'] > prev) & (growers['growth_flux'] <= hi)).sum())
    hist.append((f'({prev}, {hi}]', n))
    prev = hi
pd.DataFrame(hist, columns=['flux range', 'n_growers'])

,flux range,n_growers
0,"(0, 1]",0
1,"(1, 5]",8
2,"(5, 10]",25
3,"(10, 25]",309
4,"(25, 50]",1291
5,"(50, 75]",1135
6,"(75, 100]",693


---
## Report: `reports/CHARACTERISTICS.md`

# Part 1 — Grower vs Non-grower characteristics

Growers: 3461    Non-growers: 2222

| metric | grower median | grower mean | non-grower median | non-grower mean |
|---|---|---|---|---|
| n_metabolites | 158.0 | 156.6 | 125.0 | 124.3 |
| n_reactions | 175.0 | 171.7 | 118.0 | 119.8 |
| n_genes | 189.0 | 193.7 | 101.0 | 106.4 |
| n_exchanges_total | 27.0 | 26.7 | 23.0 | 23.3 |
| n_exchanges_open | 19.0 | 19.1 | 16.0 | 16.1 |


## Grower biomass flux distribution
- (0, 1]: 0
- (1, 5]: 8
- (5, 10]: 25
- (10, 25]: 309
- (25, 50]: 1291
- (50, 75]: 1135
- (75, 100]: 693